In [1]:
from types import coroutine

import numpy as np
import torch
import matplotlib.pyplot as plt

from torch.utils import data
from torchvision import datasets,transforms, utils

In [2]:
transform = transforms.Compose([transforms.ToTensor()])

In [3]:
tr_ds = datasets.FashionMNIST(root='./data/',
                      train=True,
                      download=True,
                      transform=transform)

In [4]:
tt_ds = datasets.FashionMNIST(root='./data/',
                      train=False,
                      download=True,
                      transform=transform)

In [5]:
tr_ds

Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: ./data/
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
           )

In [6]:
tr_ds_loader = data.DataLoader(dataset=tr_ds,batch_size=16)
tt_ds_loader = data.DataLoader(dataset=tt_ds,batch_size=16)

In [7]:
ds = iter(tr_ds_loader)
img, label = next(ds)

In [8]:
img.shape, label.shape

(torch.Size([16, 1, 28, 28]), torch.Size([16]))

In [9]:
label

tensor([9, 0, 0, 3, 0, 2, 7, 2, 5, 5, 0, 9, 5, 5, 7, 9])

In [10]:
labels={
0: "티셔츠/탑 (T-shirt/top)",
1: "트라우저 (Trouser)",
2: "풀오버 (Pullover)",
3: "드레스 (Dress)",
4: "코트 (Coat)",
5: "샌들 (Sandal)",
6: "셔츠 (Shirt)",
7: "스니커즈 (Sneaker)",
8: "가방 (Bag)",
9: "앵클 부츠 (Ankle boot)"
}

In [11]:
idx = label[0].item()
idx

9

In [12]:
labels[idx]

'앵클 부츠 (Ankle boot)'

In [13]:
from torchvision import transforms, datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [43]:
USE_MPS = torch.backends.mps.is_available()
DEVICE = torch.device("cuda" if USE_MPS else "cpu")

In [15]:
# 2. 작업준비
BATCH_SIZE = 32

In [16]:
# 1. 데이터 준비
transforms.Compose([
    transforms.ToTensor()
]) # 데이터 사용 방식 내용 결정
tr_ds = datasets.FashionMNIST(
    root='./data/',
    train=True,
    download=True,
    transform=transform
)
tt_ds = datasets.FashionMNIST(
    root='./data/',
    train=False,
    download=True,
    transform=transform
)
tr_ds_loader = torch.utils.data.DataLoader(
    dataset=tr_ds,
    batch_size=BATCH_SIZE,
    shuffle=True
)
tt_ds_loader = torch.utils.data.DataLoader(
    dataset=tt_ds,
    batch_size=BATCH_SIZE,
    shuffle=True
)

In [17]:
for i in tt_ds_loader:
    print('x: ',i[0].shape)
    print('y: ',i[1].shape)
    break

x:  torch.Size([32, 1, 28, 28])
y:  torch.Size([32])


In [18]:
# 3. 모델 설계
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
    def forward(self, x):
        x = x.view(-1, 784)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [19]:
DEVICE

device(type='mps')

In [20]:
m = Model().to(DEVICE)
opt = optim.SGD(m.parameters(), lr = 0.01)

In [21]:
# 학습 함수 설계
def train(m, tr_ds_loader, opt):
    m.train() # 학습모드
    for i,(data,target) in enumerate(tr_ds_loader):
        data, target = data.to(DEVICE), target.to(DEVICE)
        opt.zero_grad()
        output = m(data)
        loss = F.cross_entropy(output, target) # 기울기 도출
        loss.backward()
        opt.step()

In [22]:
# 예측 및 검정
def evaluate(m, tt_ds_loader):
    m.eval() # 추론모드
    test_loss = 0
    correct = 0
    with torch.no_grad(): # 회차별로 메모리 정리
        for data, target in tt_ds_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            output = m(data)

            test_loss += F.cross_entropy(output, target, reduction='sum').item()
            pred = output.max(1, keepdim=True)[1]
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(tt_ds_loader.dataset)
    test_accuracy = correct / len(tt_ds_loader.dataset)*100.
    return test_loss, test_accuracy

In [23]:
EPOCHS = 10
for i in range(1, EPOCHS+1):
    train(m, tr_ds_loader, opt)
    test_loss, test_accuracy = evaluate(m, tt_ds_loader)

    print(f'epoch: {i}, test_loss: {test_loss:.3f}, test_accuracy: {test_accuracy:.3f}')

epoch: 1, test_loss: 0.659, test_accuracy: 76.590
epoch: 2, test_loss: 0.552, test_accuracy: 80.270
epoch: 3, test_loss: 0.508, test_accuracy: 81.440
epoch: 4, test_loss: 0.474, test_accuracy: 83.080
epoch: 5, test_loss: 0.463, test_accuracy: 83.350
epoch: 6, test_loss: 0.440, test_accuracy: 84.670
epoch: 7, test_loss: 0.489, test_accuracy: 83.150
epoch: 8, test_loss: 0.413, test_accuracy: 85.310
epoch: 9, test_loss: 0.412, test_accuracy: 85.110
epoch: 10, test_loss: 0.400, test_accuracy: 85.780


In [24]:
# m.eval()
# with torch.no_grad(): # 속도 향상, 메모리 정리
#     m(data)

# 데이터를 로드하여 모델 층을 완전 연결 층 구조로 4층 쌓은 DNN 구조를 설계하고 학습 후 모델을 예측 및 검증 하시오


In [45]:
# 임포트
from torchvision import transforms, datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# mps를 우선 사용
USE_MPS = torch.backends.mps.is_available()
DEVICE = torch.device("mps" if USE_MPS else "cpu")

# 배치 사이즈 설정
BATCH_SIZE = 128

# 데이터 타입 변환
transform = transforms.Compose([transforms.ToTensor()])

# 데이터 준비(tr, tt)
tr_ds = datasets.FashionMNIST(
    root='./data/',
    # train 데이터로 지정
    train=True,
    # 만약 데이터가 없다면 다운로드
    download=False,
    # 데이터 타입을 tensor로 변환
    transform=transform
)
tt_ds = datasets.FashionMNIST(
    root='./data/',
    # test 데이터로 지정
    train=False,
    # 만약 데이터가 없다면 다운로드
    download=False,
    transform=transform
)
tr_ds_loader = torch.utils.data.DataLoader(
    dataset=tr_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
)
tt_ds_loader = torch.utils.data.DataLoader(
    dataset=tt_ds,
    batch_size=BATCH_SIZE,
    shuffle=True
)

# 모델 설계
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 10)
    def forward(self, x):
        x = x.view(-1, 784)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x

# 모델 인스턴스 생성 및 장치로 이동
model = Model().to(DEVICE)

# 옵티마이저 세팅
opt = optim.SGD(model.parameters(), lr=0.001)

# 학습 함수 설정 ,일종의 컴파일
def train(model, tr_ds_loader, opt):
    model.train() # 학습모드, 지정을 해줘야 학습이 가능
    for i,(data,target) in enumerate(tr_ds_loader):
        data, target = data.to(DEVICE), target.to(DEVICE)
        opt.zero_grad() # 기울기를 0으로 설정
        output = model(data)
        # 파이썬 내부의 cross_entropy 내부에 softmax가 내장되어 있다
        loss = F.cross_entropy(output, target) # 기울기 도출
        loss.backward()
        opt.step()

# 예측 및 검정 함수 설정(오차 계산 시각화)
def evaluate(model, tt_ds_loader):
    model.eval() # 추론모드, 지정을 해줘야 예측이 가능
    test_loss = 0
    correct = 0
    with torch.no_grad(): # 회차별로 메모리 정리, 불필요한 연산 하지 않도록
        for data, target in tt_ds_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            output = model(data)

            # 로스를 cross_entropy을 기준으로 설정
            test_loss += F.cross_entropy(output, target, reduction='sum').item()
            pred = output.max(1, keepdim=True)[1]
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(tt_ds_loader.dataset)
    test_accuracy = correct / len(tt_ds_loader.dataset)*100.
    return test_loss, test_accuracy

# 최종 함수 매 epoch 마다 학습을 반복하도록 설정하고, loss, accuracy 출력(학습 진행)
EPOCHS = 10
for i in range(1, EPOCHS+1):
    train(model, tr_ds_loader, opt)
    test_loss, test_accuracy = evaluate(model, tt_ds_loader)

    print(f'epoch: {i}, test_loss: {test_loss:.3f}, test_accuracy: {test_accuracy:.3f}')

epoch: 1, test_loss: 2.295, test_accuracy: 10.120
epoch: 2, test_loss: 2.286, test_accuracy: 16.560
epoch: 3, test_loss: 2.275, test_accuracy: 23.800
epoch: 4, test_loss: 2.261, test_accuracy: 29.850
epoch: 5, test_loss: 2.242, test_accuracy: 35.900
epoch: 6, test_loss: 2.214, test_accuracy: 36.010
epoch: 7, test_loss: 2.172, test_accuracy: 32.990
epoch: 8, test_loss: 2.111, test_accuracy: 30.480
epoch: 9, test_loss: 2.027, test_accuracy: 28.730
epoch: 10, test_loss: 1.918, test_accuracy: 32.770


In [ ]:
# 예측 및 검증
py = model(data)
F.log_softmax(py)

In [32]:
# 0. 작업 준비(이미지 처리)
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import transforms, datasets

import numpy as np
import matplotlib.pyplot as plt

In [33]:
BATCH_SIZE = 60000
tr_ds_loader = torch.utils.data.DataLoader(
    dataset=tr_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
)
img,_ = next(iter(tr_ds_loader))
img.shape

torch.Size([60000, 1, 28, 28])

In [34]:
img.mean(),img.std()

(tensor(0.2860), tensor(0.3530))

In [35]:
# 데이터 수정 (노이즈 삽입)
# 1. 데이터 준비
# 데이터를 원하는 형태로 불러오겠다(노이즈 추가 가능)
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(), # 데이터 증강 (노이즈 삽입)
    transforms.ToTensor(), # 입력 데이터 정리
    transforms.Normalize((0.2860,),(0.3530,)) # 반드시 텐서화 후 노멀라이즈 실행
])

tr_ds_loader = torch.utils.data.DataLoader(
    datasets.FashionMNIST('./data/',
        train=True,
        download=False,
        transform=transform
    ),
    batch_size=BATCH_SIZE,
    shuffle=True
)
tt_ds_loader = torch.utils.data.DataLoader(
    datasets.FashionMNIST('./data/',
        train=False,
        transform=transform
    ),
    batch_size=BATCH_SIZE,
    shuffle=True
)

In [36]:
x = 0
def f(x):
    return x+1
for i in [1,2,3]:
    x = f(x)
    print(x)

1
2
3


In [37]:
class DNN_Model(nn.Module):
    def __init__(self, input_n, hidden_ns, output_n, dropout_p=0.2):
        super().__init__()
        self.fc_in = nn.Linear(input_n, hidden_ns[0])
        # [BUG FIX] 파이썬 기본 list 대신 nn.ModuleList를 사용해야 모델의 파라미터로 정상 등록되어 학습이 진행됩니다.
        self.fc_h_l = nn.ModuleList([nn.Linear(hidden_ns[i], hidden_ns[i+1]) for i in range(len(hidden_ns)-1)])
        self.fc_out = nn.Linear(hidden_ns[-1], output_n)

        self.dropout_p = dropout_p
        self.input_n = input_n
        self.hidden_ns = hidden_ns
        self.output_n = output_n

    def forward(self, x):
        # 벡터화
        x = x.view(-1, self.input_n)
        # 입력계층 연산
        x = F.relu(self.fc_in(x))
        x = F.dropout(x, training=self.training, p=self.dropout_p)
        # 은닉계층 연산
        for i in range(len(self.fc_h_l)):
            x = F.relu(self.fc_h_l[i](x))
            x = F.dropout(x, training=self.training, p=self.dropout_p)
        # 출력계층 연산
        out = self.fc_out(x)
        return out

In [38]:
model = DNN_Model(784, [256, 128, 68], 10).to(DEVICE)
opt = optim.SGD(model.parameters(), lr=0.01)

In [39]:
def train(model, tr_ds_loader, opt):
    model.train()
    for i,(x,y) in enumerate(tr_ds_loader):
        data, target = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        py = model(data)
        loss = F.cross_entropy(py, target)
        loss.backward()
        opt.step()

In [40]:
def evaluate(model, tt_ds_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in tt_ds_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            py = model(data)
            test_loss += F.cross_entropy(py, target, reduction='sum').item()
            pred = py.max(1, keepdim=True)[1]
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(tt_ds_loader.dataset)
    test_accuracy = correct / len(tt_ds_loader.dataset)*100.
    return test_loss, test_accuracy

In [47]:
EPOCHS = 10
for i in range(1, EPOCHS+1):
    train(model, tr_ds_loader, opt)
    test_loss, test_accuracy = evaluate(model, tt_ds_loader)

    print(f'epoch: {i}, test_loss: {test_loss:.3f}, test_accuracy: {test_accuracy:.3f}')

epoch: 1, test_loss: 1.785, test_accuracy: 41.360
epoch: 2, test_loss: 1.650, test_accuracy: 43.890
epoch: 3, test_loss: 1.530, test_accuracy: 48.830
epoch: 4, test_loss: 1.423, test_accuracy: 53.610
epoch: 5, test_loss: 1.332, test_accuracy: 55.900
epoch: 6, test_loss: 1.256, test_accuracy: 56.980
epoch: 7, test_loss: 1.194, test_accuracy: 57.940
epoch: 8, test_loss: 1.143, test_accuracy: 59.790
epoch: 9, test_loss: 1.099, test_accuracy: 59.730
epoch: 10, test_loss: 1.060, test_accuracy: 61.250


In [ ]:
@torch.no_grad() #서식
def evaluate(model, tt_ds_loader):
    model.eval()
    test_loss = 0
    correct = 0
    for data, target in tt_ds_loader:
        data, target = data.to(DEVICE), target.to(DEVICE)
        py = model(data)
        test_loss += F.cross_entropy(py, target, reduction='sum').item()
        pred = py.max(1, keepdim=True)[1]
        correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(tt_ds_loader.dataset)
    test_accuracy = correct / len(tt_ds_loader.dataset)*100.
    return test_loss, test_accuracy